# Day 2 — Joins, Set Logic, Subqueries and Advanced Relationships

**Target:** 3–4 years experience SQL hands-on practice.

**Database:** MySQL 8.x. The tables/data are created in a separate setup SQL file; this notebook focuses on syllabus, commands, explanations, practice queries, assignments, and solutions.


## Learning method

1. Run each query manually in MySQL Workbench/CLI.
2. Before executing, predict the result.
3. Change one condition and rerun.
4. Check execution plans where applicable.
5. Complete assignments without looking at the solution first.


## Important execution commands

```sql
SHOW DATABASES;
USE company_sql_practice;
SHOW TABLES;
SELECT DATABASE();
DESCRIBE employees;
```

The setup file contains the five-table schema and large sample dataset.


## Syllabus

- INNER JOIN
- LEFT JOIN and anti-joins
- RIGHT JOIN
- CROSS JOIN
- SELF JOIN / employee-manager hierarchy
- Joining 3, 4 and 5 tables
- Join cardinality and duplicate explosion
- Composite join conditions
- JOIN vs EXISTS
- Subqueries and derived tables
- CTEs for multi-step business logic
- UNION vs UNION ALL
- INTERSECT / EXCEPT where supported
- Relational division patterns
- Data quality checks using joins


## Practice Queries

Each query is intentionally written as a workplace-style example. Read the explanation before changing it.


### Practice 1

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- 4-table join: employee -> department -> project -> assignment
SELECT e.employee_name, d.department_name, p.project_name,
       ep.role_name, ep.allocation_pct
FROM employees e
JOIN departments d ON d.department_id = e.department_id
JOIN employee_projects ep ON ep.employee_id = e.employee_id
JOIN projects p ON p.project_id = ep.project_id
ORDER BY d.department_name, e.employee_name;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 2

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- LEFT JOIN: departments with no employees
SELECT d.department_id, d.department_name
FROM departments d
LEFT JOIN employees e ON e.department_id = d.department_id
WHERE e.employee_id IS NULL;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 3

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- SELF JOIN: employee-manager relationship
SELECT e.employee_name AS employee,
       m.employee_name AS manager
FROM employees e
LEFT JOIN employees m ON m.employee_id = e.manager_id
ORDER BY manager, employee;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 4

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Anti-join: customers with no orders
SELECT c.customer_id, c.customer_name
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 5

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Multi-table revenue report
SELECT c.customer_name,
       COUNT(DISTINCT o.order_id) AS order_count,
       SUM(oi.quantity * oi.unit_price) AS revenue
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY c.customer_id, c.customer_name
ORDER BY revenue DESC;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 6

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- JOIN + derived table
SELECT e.employee_name, e.salary, ds.avg_salary
FROM employees e
JOIN (
    SELECT department_id, AVG(salary) AS avg_salary
    FROM employees
    GROUP BY department_id
) ds ON ds.department_id = e.department_id
WHERE e.salary > ds.avg_salary;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


## Assignments

1. Create a 5-table join returning employee, department, project, customer and revenue information.
2. Find managers with at least 3 direct reports.
3. Find customers who placed orders in every month of a selected year.
4. Find products never ordered.
5. Find employees assigned to more than 2 projects.
6. Demonstrate a join that causes duplicate rows, diagnose why, then fix it.


## Solutions

Try the assignment first. Then compare with the reference pattern below. Adapt names/filters to your dataset.


### Solution 1


In [ ]:
SELECT e.employee_name, d.department_name, p.project_name,
       c.customer_name,
       SUM(oi.quantity * oi.unit_price) revenue
FROM employees e
JOIN departments d ON d.department_id=e.department_id
JOIN employee_projects ep ON ep.employee_id=e.employee_id
JOIN projects p ON p.project_id=ep.project_id
JOIN customers c ON c.customer_id=p.customer_id
JOIN orders o ON o.customer_id=c.customer_id
JOIN order_items oi ON oi.order_id=o.order_id
GROUP BY e.employee_id,e.employee_name,d.department_name,p.project_name,c.customer_name;


### Solution 2


In [ ]:
SELECT m.employee_id, m.employee_name, COUNT(*) direct_reports
FROM employees e
JOIN employees m ON m.employee_id=e.manager_id
GROUP BY m.employee_id,m.employee_name
HAVING COUNT(*) >= 3;


### Solution 3


In [ ]:
SELECT c.customer_id, c.customer_name
FROM customers c
JOIN orders o ON o.customer_id=c.customer_id
WHERE YEAR(o.order_date)=2026
GROUP BY c.customer_id,c.customer_name
HAVING COUNT(DISTINCT MONTH(o.order_date))=12;


### Solution 4


In [ ]:
SELECT p.project_id, p.project_name
FROM projects p
LEFT JOIN employee_projects ep ON ep.project_id=p.project_id
WHERE ep.project_id IS NULL;


### Solution 5


In [ ]:
SELECT employee_id, COUNT(*) project_count
FROM employee_projects
GROUP BY employee_id
HAVING COUNT(*) > 2;


### Solution 6


In [ ]:
-- Diagnose duplicates by checking the grain of each joined table.
SELECT o.order_id, COUNT(*) AS rows_after_join
FROM orders o
JOIN order_items oi ON oi.order_id=o.order_id
GROUP BY o.order_id
HAVING COUNT(*) > 1;


## Interview Questions & Answers

**Q: WHERE vs HAVING?**  
A: WHERE filters rows before grouping; HAVING filters grouped results.

**Q: RANK vs DENSE_RANK?**  
A: Both assign equal ranks to ties; RANK leaves gaps after ties, DENSE_RANK does not.

**Q: Why can joins create duplicates?**  
A: A one-to-many relationship legitimately creates multiple result rows per parent. First define the intended row grain, then aggregate/deduplicate as required.

**Q: Cursor or set-based SQL?**  
A: Prefer set-based operations for bulk relational transformations; use cursors when row-by-row procedural behavior is genuinely required.

**Q: What does EXPLAIN provide?**  
A: It exposes the optimizer's planned access path, including table access, candidate indexes, selected index, estimated rows and other plan details.


## 5-Day Deliverable Checklist

- [ ] All practice queries executed manually
- [ ] All assignments solved without copying
- [ ] Solutions compared and rewritten
- [ ] At least 10 query variations created
- [ ] At least 5 window-function problems solved
- [ ] At least 3 stored procedures created
- [ ] At least 1 cursor + handler implemented
- [ ] Transactions tested with COMMIT and ROLLBACK
- [ ] EXPLAIN used before/after an index
- [ ] Final capstone query set completed
